Import big matrix

In [1]:
import pandas as pd

bm_og = pd.read_csv(
    '/home/sulay/deep-linear-bandits/kuairec/data/big_matrix.csv',
        usecols=[
            'user_id',
            'video_id',
            'watch_ratio'
        ]
    ).sort_values(by=['user_id', 'video_id'])

bm_og

,user_id,video_id,watch_ratio
312,0,42,1.098951
292,0,67,2.759635
275,0,80,1.188017
61,0,110,1.408627
318,0,128,1.281867
...,...,...,...
12530627,7175,10552,0.147431
12530710,7175,10572,0.293611
12530711,7175,10572,0.293611
12530717,7175,10589,0.560359


In [2]:
bm_og.duplicated(subset=['user_id', 'video_id']).sum()

np.int64(2229837)

In [3]:
bm_f = bm_og.copy()

bm_f = bm_f.groupby(by=['user_id', 'video_id'], as_index=False).max()

bm_f

,user_id,video_id,watch_ratio
0,0,42,1.098951
1,0,67,2.759635
2,0,80,1.188017
3,0,110,1.408627
4,0,128,1.281867
...,...,...,...
10300964,7175,10519,1.107321
10300965,7175,10552,0.147431
10300966,7175,10572,0.293611
10300967,7175,10589,0.560359


In [4]:
bm_f.groupby(by='user_id').count().sort_values(by='watch_ratio')

,video_id,watch_ratio
user_id,,
3116,80,80
4604,125,125
3935,151,151
3175,155,155
268,156,156
...,...,...
6565,3341,3341
816,3348,3348
5412,3353,3353


In [5]:
bm_f[bm_f.watch_ratio < 2.0].groupby(by='user_id').count().sort_values(by='watch_ratio')

,video_id,watch_ratio
user_id,,
3116,63,63
4604,102,102
3935,121,121
3175,133,133
4812,138,138
...,...,...
6565,3295,3295
2735,3303,3303
5412,3308,3308


In [6]:
bm_f.groupby(by='video_id').count().sort_values(by='watch_ratio')

,user_id,watch_ratio
video_id,,
11,1,1
4943,1,1
40,1,1
43,1,1
10705,1,1
...,...,...
1507,5168,5168
8145,5175,5175
1037,5211,5211


Filter for the strong positive interactions to train on

In [7]:
bm = bm_f.copy()

bm = bm[bm.watch_ratio >= 2.0]

bm

,user_id,video_id,watch_ratio
1,0,67,2.759635
6,0,133,2.458447
10,0,152,2.326087
11,0,154,4.353647
15,0,171,33.276021
...,...,...,...
10300855,7175,9811,2.232981
10300862,7175,9841,2.167814
10300910,7175,10130,15.828342
10300955,7175,10408,9.241597


In [8]:
liked_counts = bm.groupby(by='user_id')['user_id'].count()

liked_counts

user_id
0       258
1       166
2        38
3       163
4        51
       ... 
7171    134
7172    271
7173     95
7174     37
7175    223
Name: user_id, Length: 7175, dtype: int64

In [9]:
liked_counts.describe()

count    7175.000000
mean      117.967108
std       101.901022
min         1.000000
25%        43.000000
50%        88.000000
75%       162.000000
max      1002.000000
Name: user_id, dtype: float64

In [10]:
bm_f[bm_f.watch_ratio < 2.0]

,user_id,video_id,watch_ratio
0,0,42,1.098951
2,0,80,1.188017
3,0,110,1.408627
4,0,128,1.281867
5,0,130,0.079565
...,...,...,...
10300964,7175,10519,1.107321
10300965,7175,10552,0.147431
10300966,7175,10572,0.293611
10300967,7175,10589,0.560359


Split into training & validation

In [11]:
from sklearn.model_selection import train_test_split

low = bm.groupby('user_id')['user_id'].transform('count') < 5

nonlow = bm[~low]
bm_train, bm_val = train_test_split(
    nonlow,
    train_size=0.8,
    shuffle=True,
    stratify=nonlow['user_id']
)
bm_train = pd.concat([bm_train, bm[low]])

bm_train, bm_val

(         user_id  video_id  watch_ratio
 8530735     5921      8496     5.337429
 5775789     4011      8629     2.621893
 5586493     3878      7528     2.922558
 732340       487       229     3.697237
 9641905     6718      4530     2.396023
 ...          ...       ...          ...
 9176402     6384      3632     3.930262
 9555229     6658      3400     2.603915
 9555261     6658      4374     2.519602
 9555297     6658      5047     2.600979
 9555519     6658     10653     2.045629
 
 [677144 rows x 3 columns],
          user_id  video_id  watch_ratio
 1001716      676      2766     4.127204
 7721170     5341      2713     2.673097
 7055299     4881      5793     2.984630
 5467463     3793      2588     3.041460
 946152       636      2311     2.347667
 ...          ...       ...          ...
 9019977     6277      6734     2.537318
 1377255      933       799     2.058015
 9238344     6429      3245     6.109862
 7944489     5500     10014     7.107792
 5573969     3870      3523

Convert to PyTorch dataset format

In [74]:
from torch.utils.data import Dataset
import torch
import numpy as np

K = 5

class KRDataset(Dataset):
    def __init__(self, data):
        self.user_ids = data['user_id'].to_numpy()
        self.item_ids = data['video_id'].to_numpy()
        
        # Compute user positive sets for rejection sampling negatives
        self.positives = [set() for _ in range(7176)]
        for i in range(len(self.user_ids)):
            self.positives[self.user_ids[i]].add(self.item_ids[i])

    def __len__(self):
        return len(self.user_ids)
    
    def __getitem__(self, idx):
        # Rejection sample negatives
        negatives = []
        while len(negatives) < K:
            n = np.random.randint(0, 7176)
            if n not in self.positives[self.user_ids[idx]]: negatives.append(n)

        return {
            'user_id': self.user_ids[idx],
            'pos_item_id': self.item_ids[idx],
            'neg_item_ids': torch.tensor(negatives)
        }

bm_t = KRDataset(bm_train)
bm_v = KRDataset(bm_val)

print(len(bm_t))
print(len(bm_v))

677144
169270


Set up two-tower

In [75]:
from torch import nn

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()

        self.u = nn.Embedding(7176, 32)
        self.i = nn.Embedding(10728, 32)

    def forward(self, user_id, pos_item_id, neg_item_ids):
        u_emb = self.u(user_id) # BATCH_SIZE x 1 -> BATCH_SIZE x EMB_SIZE
        pos_emb = self.i(pos_item_id) # BATCH_SIZE x 1 -> BATCH_SIZE x EMB_SIZE
        neg_embs = self.i(neg_item_ids) # BATCH_SIZE x NEGSAMPLES_NUM x 1 -> BATCH_SIZE x NEGSAMPLES_NUM x EMB_SIZE

        dots = (u_emb * pos_emb).sum(1, keepdim=True) # dot product between user and item for each batch -> BATCH_SIZE x 1

        u_emb = u_emb.unsqueeze(1) # BATCH_SIZE x EMB_SIZE -> BATCH_SIZE x 1 x EMB_SIZE
        neg_embs = neg_embs.transpose(1,2) # BATCH_SIZE x NEGSAMPLES_NUM x EMB_SIZE -> BATCH_SIZE x EMB_SIZE x NEGSAMPLES_NUM
        bmm = torch.bmm(u_emb, neg_embs) # matmul for each batch to get dot prod of user with all negatives -> BATCH_SIZE x 1 x NEGSAMPLES_NUM
        bmm = bmm.squeeze(1) # -> BATCH_SIZE x NEGSAMPLES_NUM

        # now combine the pos & neg for each batch
        return torch.cat([dots, bmm], dim=1) # BATCH_SIZE x (1 + NEGSAMPLES_NUM)

model = TwoTower().to(device)

model

TwoTower(
  (u): Embedding(7176, 32)
  (i): Embedding(10728, 32)
)

Train the two-tower and see validation loss each epoch

In [76]:
from torch.utils.data import DataLoader
from tqdm import tqdm

bm_train_ldr = DataLoader(bm_t, batch_size=512, shuffle=True)
bm_val_ldr = DataLoader(bm_v, batch_size=512, shuffle=True)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

epochs=50
for epoch in range(epochs):
    model.train()
    total_loss_t = 0
    for batch in tqdm(bm_train_ldr):
        opt.zero_grad()

        logits = model(
            batch['user_id'].to(device),
            batch['pos_item_id'].to(device),
            batch['neg_item_ids'].to(device)
        )
        target = torch.zeros(logits.size(0), dtype=torch.long, device=device)

        loss = loss_fn(logits, target)

        loss.backward()
        opt.step()

        total_loss_t += loss.item()
    
    print(f"End of epoch {epoch} average batch training loss: {total_loss_t / len(bm_train_ldr)}")

100%|██████████| 1323/1323 [00:19<00:00, 67.96it/s]


End of epoch 0 average batch training loss: 6.852486653302712


100%|██████████| 1323/1323 [00:20<00:00, 63.02it/s]


End of epoch 1 average batch training loss: 5.870236400754183


100%|██████████| 1323/1323 [00:18<00:00, 71.42it/s]


...

End of epoch 47 average batch training loss: 0.3504718702711666


100%|██████████| 1323/1323 [00:19<00:00, 68.75it/s]


End of epoch 48 average batch training loss: 0.34912349965116424


100%|██████████| 1323/1323 [00:19<00:00, 67.75it/s]

End of epoch 49 average batch training loss: 0.34652378510906223
